In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

sales = pd.read_csv("retail_store_sales.csv")

sales.head()

# 15. Cleaning Pipeline Cheat Sheet

| Cleaning task | Pandas operation | Example |
|---|---|---|
| Find duplicate rows | `duplicated()` | `df.duplicated().sum()` |
| Remove duplicate rows | `drop_duplicates()` | `df.drop_duplicates()` |
| Remove duplicates using selected columns | `drop_duplicates(subset=...)` | `df.drop_duplicates(subset=["ID"])` |
| Rename columns | `rename()` | `df.rename(columns={"Old":"New"})` |
| Convert data type | `astype()` | `df["Amount"].astype(float)` |
| Convert messy numbers | `pd.to_numeric()` | `pd.to_numeric(..., errors="coerce")` |
| Convert dates | `pd.to_datetime()` | `pd.to_datetime(df["Date"])` |
| Remove spaces | `.str.strip()` | `df["City"].str.strip()` |
| Lowercase | `.str.lower()` | `df["City"].str.lower()` |
| Uppercase | `.str.upper()` | `df["City"].str.upper()` |
| Title Case | `.str.title()` | `df["City"].str.title()` |
| Search text | `.str.contains()` | `df["Item"].str.contains("Coffee")` |
| Replace text | `.str.replace()` | `df["City"].str.replace(...)` |

# PART 4 — TIME SERIES ANALYSIS

## 35. Why does time matter in data analysis?

So far, most of our questions have looked like:

- How much did each category sell?
- How many transactions used each payment method?
- How do online and in-store sales compare?

But businesses also ask questions such as:

> How much did we sell each day?
>
> Which day of the week has the most transactions?
>
> How did sales change from month to month?
>
> What is the average sales level over the last 7 days?

These are **time-based questions**.

When a dataset contains dates or timestamps, Pandas gives us special tools for working with time.

The main workflow is:

```text
DATE/TIME TEXT
      ↓
pd.to_datetime()
      ↓
DATETIME VALUES
      ↓
.dt.year / .dt.day_name() / other components
      ↓
DatetimeIndex
      ↓
resample() / rolling()
      ↓
TIME-BASED INSIGHTS
```

## 36. First inspect `Transaction Date`

Our retail dataset already contains a `Transaction Date` column.

Before doing anything with dates, inspect it.

In [ ]:
sales[["Transaction Date"]].head()

In [ ]:
sales["Transaction Date"].dtype

### Why are we checking the data type?

A date can be stored as **text** (`object`/string) rather than as an actual date.

For example:

```text
"2022-01-15"
```

looks like a date to us, but Pandas may simply see it as characters.

If Pandas does not recognize the values as dates, time-based operations such as extracting the year or resampling will not work as expected.

So our first job is to convert the column into a real datetime type.

## 37. Parsing timestamps with `pd.to_datetime()`

`pd.to_datetime()` converts values into Pandas' datetime representation.

Think of it as:

> **"Pandas, please understand these values as dates/times rather than ordinary text."**

In [ ]:
sales["Transaction Date"] = pd.to_datetime(
    sales["Transaction Date"],
    errors="coerce"
)

sales[["Transaction Date"]].head()

### Why `errors="coerce"`?

If Pandas encounters a value that cannot be interpreted as a date, `errors="coerce"` turns that value into `NaT` (Not a Time) instead of stopping the entire conversion with an error.

This is useful when working with messy real-world data.

After conversion, check for values that failed:

In [ ]:
sales["Transaction Date"].isna().sum()

### Important distinction

`NaN` is commonly used for missing numerical values.

`NaT` is used for missing or invalid **datetime** values.

So after parsing dates, always check whether any `NaT` values were created.

## 38. Extracting date components with `.dt`

Once a column has been converted to datetime, Pandas gives us the `.dt` accessor.

The `.dt` accessor lets us ask questions about individual parts of a date.

For example:

```python
sales["Transaction Date"].dt.year
```

means:

> "From every datetime value, give me the year."

This is similar to how we use `.str` for string operations.

- `.str` → work with text
- `.dt` → work with dates/times

In [ ]:
sales["Year"] = sales["Transaction Date"].dt.year

sales[["Transaction Date", "Year"]].head()

### Extracting the day name

We can also ask Pandas for the name of the day:

In [ ]:
sales["Day Name"] = sales["Transaction Date"].dt.day_name()

sales[["Transaction Date", "Day Name"]].head()

### Other useful datetime components

Pandas provides many `.dt` attributes and methods.

| Expression | Gives us |
|---|---|
| `.dt.year` | year |
| `.dt.month` | month number |
| `.dt.day` | day of month |
| `.dt.day_name()` | Monday, Tuesday, etc. |
| `.dt.month_name()` | January, February, etc. |
| `.dt.dayofweek` | weekday number, Monday = 0 |
| `.dt.quarter` | quarter number |

The important idea is that **we extract information from a datetime value instead of treating the date as plain text**.

In [ ]:
sales["Month"] = sales["Transaction Date"].dt.month
sales["Month Name"] = sales["Transaction Date"].dt.month_name()
sales["Quarter"] = sales["Transaction Date"].dt.quarter

sales[[
    "Transaction Date",
    "Year",
    "Month",
    "Month Name",
    "Quarter",
    "Day Name"
]].head()

## 39. Using extracted dates for analysis

Now that we have date components, we can combine them with what we learned yesterday.

For example:

> **What are total sales by day of the week?**

This is a familiar `groupby()` problem — the only new thing is that the grouping variable comes from a datetime column.

In [ ]:
day_sales = (
    sales.groupby("Day Name")["Total Spent"]
    .sum()
    .reset_index()
)

day_sales

### A better weekday order

There is one issue with the previous result: Pandas may display the day names alphabetically rather than in calendar order.

We can create an ordered categorical variable so that the days appear Monday → Sunday.

In [ ]:
day_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

sales["Day Name"] = pd.Categorical(
    sales["Day Name"],
    categories=day_order,
    ordered=True
)

sales.groupby("Day Name", observed=True)["Total Spent"].sum()

## 40. Setting a `DatetimeIndex`

A **DatetimeIndex** is an index whose values are dates/times.

Why is this useful?

Because Pandas' time-series tools are designed to work especially well when time is the index.

We can set `Transaction Date` as the index with:

```python
sales.set_index("Transaction Date")
```

This does not automatically modify the original DataFrame unless we assign the result back or use `inplace=True`.

In [ ]:
sales_time = sales.set_index("Transaction Date")

sales_time.head()

In [ ]:
type(sales_time.index)

### Notice what changed

Before:

```text
0    transaction 1
1    transaction 2
2    transaction 3
```

After setting the index:

```text
Transaction Date
2022-01-01    transaction 1
2022-01-01    transaction 2
2022-01-02    transaction 3
```

The date is now the row label.

This is the structure we want before using tools such as `resample()`.

# PART 5 — RESAMPLING & WINDOWS

## 41. What is resampling?

Imagine we have many observations recorded at one frequency and we want to analyze them at another frequency.

For example:

```text
Daily sales
Monday
Tuesday
Wednesday
Thursday
Friday
...

        ↓ resample

Weekly sales
Week 1
Week 2
Week 3
...
```

`resample()` changes the **time frequency** of a time-indexed DataFrame or Series.

Two important ideas:

- **Downsampling** → moving from a more detailed frequency to a less detailed frequency.
- **Upsampling** → moving from a less detailed frequency to a more detailed frequency.

## 42. Downsampling daily sales to weekly sales

Our transaction data contains individual transactions. Let's first create a daily sales series.

We can group the transactions by day and add up `Total Spent`.

In [ ]:
daily_sales = (
    sales_time["Total Spent"]
    .resample("D")
    .sum()
)

daily_sales.head()

### Understanding the frequency code

`"D"` means **day**.

Some useful frequency codes include:

| Code | Meaning |
|---|---|
| `"D"` | daily |
| `"W"` | weekly |
| `"ME"` | month-end |
| `"QE"` | quarter-end |
| `"YE"` | year-end |
| `"h"` | hourly |

Frequency aliases have changed slightly across Pandas versions, so it is good practice to use current aliases such as `ME`, `QE`, and `YE` for month-, quarter-, and year-end frequencies.

## 43. Weekly sales with `resample("W")`

Now let's convert our daily transaction series into weekly totals.

Read this as:

> **"Use the datetime index to put observations into weekly groups, then sum the sales in each group."**

In [ ]:
weekly_sales = (
    sales_time["Total Spent"]
    .resample("W")
    .sum()
)

weekly_sales.head()

### Why is this downsampling?

Suppose we start with:

```text
365 daily observations
```

and turn them into:

```text
52 weekly observations
```

We have moved from a **more detailed time frequency** to a **less detailed frequency**.

That is downsampling.

## 44. Resampling with different aggregation functions

`resample()` creates time-based groups. We then decide how to summarize each group.

For example, we can calculate:

- weekly total sales → `sum()`
- weekly average transaction value → `mean()`
- largest transaction each week → `max()`

In [ ]:
weekly_summary = (
    sales_time["Total Spent"]
    .resample("W")
    .agg(["sum", "mean", "max"])
)

weekly_summary.head()

### The connection to `groupby()`

This is worth emphasizing:

`groupby()` groups based on a column or columns.

`resample()` groups based on **time intervals in a DatetimeIndex**.

Conceptually:

```text
.groupby("Category")
        ↓
Group rows by category

.resample("W")
        ↓
Group rows into weekly time periods
```

Both are grouping operations — they simply group by different things.

## 45. Upsampling

Upsampling moves from a less detailed frequency to a more detailed one.

For example:

```text
Monthly data
January
February
March

        ↓ upsample

Daily data
Jan 1
Jan 2
Jan 3
...
```

A crucial point:

> **Upsampling creates additional time points; it does not magically create new information.**

The newly created dates may contain missing values until we decide how to fill them.

In [ ]:
monthly_demo = pd.Series(
    [1000, 1200, 900],
    index=pd.to_datetime(["2026-01-31", "2026-02-28", "2026-03-31"]),
    name="Monthly Sales"
)

monthly_demo

In [ ]:
monthly_daily = monthly_demo.resample("D").asfreq()

monthly_daily.head(10)

### Why do we see `NaN` values?

We started with only three observations — one for each month-end.

When we ask for daily frequency, Pandas creates the daily dates between those observations, but it does not know what sales value belongs to those new dates.

So we get missing values.

There are several ways to handle this depending on the meaning of the data. For example, `ffill()` carries the previous known value forward.

In [ ]:
monthly_daily_ffill = monthly_demo.resample("D").ffill()

monthly_daily_ffill.head(10)

### A warning about upsampling

Do not automatically fill every new time point.

If the original values represent **monthly totals**, carrying the total into every day would be misleading.

The correct filling method depends on what the original measurement means.

This is a data interpretation issue, not just a coding issue.

## 46. Rolling windows

A **rolling window** looks at a moving group of recent observations.

For example, a 7-day rolling mean asks:

> "For each day, what is the average sales value across this day and the previous 6 days?"

Visual idea:

```text
Day 1  Day 2  Day 3  Day 4  Day 5  Day 6  Day 7
  └────────────── 7-day window ──────────────┘

Then the window moves:

Day 2  Day 3  Day 4  Day 5  Day 6  Day 7  Day 8
  └────────────── 7-day window ──────────────┘
```

Unlike `resample()`, rolling windows **overlap**.

## 47. Rolling mean

Let's calculate a 7-day rolling average of daily sales.

In [ ]:
daily_sales_rolling_mean = daily_sales.rolling(window=7).mean()

daily_sales_rolling_mean.head(10)

### Why are the first values missing?

A 7-day rolling window needs **7 observations** before it can calculate the first complete 7-day mean.

So the first six positions are `NaN`.

After the seventh observation, Pandas has enough values to calculate the first window.

## 48. Rolling standard deviation

The standard deviation tells us how spread out the values are.

A rolling standard deviation therefore tells us how much sales have been varying within each moving window.

For example:

In [ ]:
daily_sales_rolling_std = daily_sales.rolling(window=7).std()

daily_sales_rolling_std.head(10)

### Mean vs standard deviation in a rolling window

For a 7-day window:

- `rolling(7).mean()` → the **average** sales across the last 7 observations.
- `rolling(7).std()` → the **spread/variability** of sales across the last 7 observations.

This can help an analyst distinguish between:

> **A changing sales level** and **a changing level of sales volatility**.

## 49. `min_periods` — allowing earlier rolling results

By default, a 7-observation window waits until it has all 7 observations.

We can change that with `min_periods`.

For example, `min_periods=1` means:

> "Calculate the rolling statistic as soon as at least one observation is available."

In [ ]:
daily_sales.rolling(
    window=7,
    min_periods=1
).mean().head(10)

### Be careful with interpretation

`min_periods=1` gives us more values, but the early values are based on smaller windows.

For example:

- Day 1 → average of 1 day
- Day 2 → average of 2 days
- Day 3 → average of 3 days
- ...
- Day 7 → average of 7 days

So the calculation is valid, but the window size is not yet consistent at the beginning.

# PART 6 — PUTTING TIME SERIES CONCEPTS TOGETHER

## 50. A complete time-series workflow

Let's combine the ideas in one realistic workflow.

### Business question

> **"How do weekly sales change over time, and what does the 4-week moving average tell us?"**

We will:

1. Convert `Transaction Date` to datetime.
2. Set it as the index.
3. Resample transactions into weekly sales.
4. Calculate a 4-week rolling mean.
5. Calculate a 4-week rolling standard deviation.

In [ ]:
time_sales = sales.copy()

time_sales["Transaction Date"] = pd.to_datetime(
    time_sales["Transaction Date"],
    errors="coerce"
)

time_sales = time_sales.set_index("Transaction Date")

weekly_sales = time_sales["Total Spent"].resample("W").sum()

weekly_analysis = pd.DataFrame({
    "weekly_sales": weekly_sales,
    "rolling_4_week_mean": weekly_sales.rolling(4).mean(),
    "rolling_4_week_std": weekly_sales.rolling(4).std()
})

weekly_analysis.head(10)

### How to read the final table

Each row represents a **week**.

- `weekly_sales` → actual sales recorded during that week.
- `rolling_4_week_mean` → average weekly sales across the current and previous 3 weeks.
- `rolling_4_week_std` → how much the weekly sales values have varied across that 4-week window.

This is much more informative than looking at one weekly number in isolation because the rolling calculations give us **context from recent weeks**.

# PART 7 — GUIDED PRACTICE

## Exercise 1 — Extract the year

Create a new column called `Transaction Year` from `Transaction Date`.

**Hint:** use `.dt.year`.

In [ ]:
# Your code here

## Exercise 2 — Extract the day name

Create a new column called `Transaction Day` containing the day name.

**Hint:** use `.dt.day_name()`.

In [ ]:
# Your code here

## Exercise 3 — Total sales by day of the week

Use the extracted day name to calculate total `Total Spent` for each day.

In [ ]:
# Your code here

## Exercise 4 — Create a DatetimeIndex

Create a new DataFrame called `time_sales` with `Transaction Date` as its index.

In [ ]:
# Your code here

## Exercise 5 — Weekly sales

Using `time_sales`, calculate total weekly sales with `resample()`.

**Hint:** use `"W"` and `sum()`.

In [ ]:
# Your code here

## Exercise 6 — 4-week rolling mean

Using your weekly sales Series, calculate a 4-week rolling mean.

In [ ]:
# Your code here

# PART 8 — CHALLENGE

## Sales trend analyst task

Imagine you are a junior data analyst at the retail company.

Your manager asks:

> "Prepare a time-based sales summary that helps me understand how sales behave over time."

Complete the following:

### A. Date preparation
- Convert `Transaction Date` to datetime.
- Check whether any invalid/missing dates remain.

### B. Date components
Create:
- `Year`
- `Month`
- `Day Name`

### C. Weekly reporting
Create a weekly sales Series containing total `Total Spent`.

### D. Rolling analysis
Add:
- 4-week rolling mean
- 4-week rolling standard deviation

### E. Interpretation
Write 3–5 sentences explaining what the weekly sales and rolling measures tell you.

Do not just describe what the code does. Interpret what the numbers mean from a business perspective.

# PART 9 — COMMON BEGINNER MISTAKES

## Mistake 1 — Treating dates as ordinary strings

```python
sales["Transaction Date"].dt.year
```

will not work properly if the column has not been converted to datetime.

**Fix:**

```python
sales["Transaction Date"] = pd.to_datetime(sales["Transaction Date"])
```

---

## Mistake 2 — Using `resample()` without a time-aware index

`resample()` expects a time-based index or an appropriate time-based selection.

A common beginner workflow is:

```python
sales.set_index("Transaction Date")
```

but forgetting to save the result.

Remember:

```python
sales_time = sales.set_index("Transaction Date")
```

---

## Mistake 3 — Confusing `resample()` and `rolling()`

They are both time-related, but they do different jobs.

- `resample("W")` → creates **weekly groups**.
- `rolling(7)` → creates a **moving 7-observation window**.

A useful memory trick:

> **Resample changes the time frequency. Rolling moves a window across the data.**

---

## Mistake 4 — Thinking upsampling creates new information

Upsampling creates new time points. It does not create real measurements that were never recorded.

Any filling or interpolation step needs to make sense for the meaning of the data.

---

## Mistake 5 — Forgetting what the rolling window represents

```python
rolling(7)
```

means **7 observations**, not automatically "7 calendar days" in every situation.

If your data has missing dates or irregular observations, this distinction matters.

# FINAL RECAP — WHAT YOU SHOULD KNOW

## Reshaping

```python
pd.pivot_table(...)
```
→ summarize and spread values across columns

```python
pd.crosstab(...)
```
→ count combinations of categorical variables

```python
df.melt(...)
```
→ convert wide data to long data

## Combining

```python
pd.merge(...)
```
→ match related records using a key

```python
pd.concat([...])
```
→ stack or attach DataFrames

## Time Series

```python
pd.to_datetime(...)
```
→ convert values into datetime

```python
df["Date"].dt.year
```
→ extract the year

```python
df["Date"].dt.day_name()
```
→ extract the day name

```python
df.set_index("Date")
```
→ make the date the index

```python
df.resample("W").sum()
```
→ summarize data into weekly periods

```python
df.rolling(7).mean()
```
→ calculate a moving 7-observation average

```python
df.rolling(7).std()
```
→ calculate moving variability

### The big picture

```text
AGGREGATION
   groupby() / agg()
        ↓
RESHAPING
   pivot / crosstab / melt
        ↓
COMBINING
   merge / concat
        ↓
TIME
   to_datetime / .dt / DatetimeIndex
        ↓
TIME-BASED GROUPING
   resample()
        ↓
TIME-BASED SMOOTHING & VARIABILITY
   rolling().mean()
   rolling().std()
```

### The most important question to keep asking

> **What does each row represent, what does each column represent, and what does my calculation mean?**

Good Pandas analysis is not about memorizing syntax. It is about choosing an operation that matches the question you are trying to answer.

# Optional Homework

Using `retail_store_sales.csv`:

1. Convert `Transaction Date` to datetime and check for invalid dates.
2. Find total sales by year.
3. Find total sales by month.
4. Find total sales by day of the week.
5. Create a weekly sales Series.
6. Calculate a 4-week rolling mean of weekly sales.
7. Calculate a 4-week rolling standard deviation of weekly sales.
8. Explain the difference between `resample()` and `rolling()` in your own words.
9. Explain the difference between downsampling and upsampling.
10. Write a short business interpretation of your rolling analysis.